[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C57_Small_Object_Detection_Course/01_why_hard/01_why_small_is_hard.ipynb)

# 01 · 定量分析：小目标到底难在哪（IoU-位移闭式解 / 临界位移 / anchor 上界 / stride 预算 / ERF / 标注噪声）

这是全课的**地基**。目标只有一个：把「小目标难」这句定性抱怨，
变成 **五组可以写在白板上的数字**。

**本 notebook 你会亲手实现：**
1. IoU 随位移衰减的 **1D / 2D 闭式解**，并用数值积分校验导数 `dIoU/dd = -4/s`
2. **临界位移** `d* = s(1-sqrt(2t/(1+t)))`，把 IoU 阈值翻译成「允许偏几个像素」
3. anchor 匹配的**上界** `(s/A)^2`，证明 8px 目标在标准配置下**一个正样本都拿不到**
4. stride 上的**格子预算**与两个小目标的**同格（混叠）概率**
5. **有效感受野**的数值实验：验证 `sigma_ERF = sqrt(2n/3)`，理论感受野按 O(n)、有效感受野按 O(sqrt(n))
6. 标注抖动对小框 IoU 分布的影响，以及由它决定的 **AP 上限**
7. 一个**渲染实验**：在固定镜头模糊下，「60」与「80」的可分性 d' 随尺寸怎么塌

> 心智模型：**IoU 是尺度不变的尺子，但检测系统的误差是尺度不变不了的。
> 用相对尺子去量绝对误差 —— 不公平就是这么来的。**

## 1 · IoU 随位移的衰减：闭式解与数值验证

两个边长为 `s` 的轴对齐正方形，预测框相对真值在 x、y 各偏 `d` 像素：

$$\mathrm{IoU}(s,d)=\frac{(s-d)^2}{2s^2-(s-d)^2}\qquad
\mathrm{IoU}_{1D}(s,d)=\frac{s-d}{s+d}$$

In [ ]:
import sys, math, platform, json
import numpy as np

print('Python', sys.version.split()[0], '|', platform.system(), platform.machine())
print('numpy ', np.__version__)
rng = np.random.default_rng(0)

def iou_xyxy(a, b):
    '''a, b: (..., 4) 的 [x1, y1, x2, y2]。逐对 IoU。'''
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    ix = np.maximum(0.0, np.minimum(a[..., 2], b[..., 2]) - np.maximum(a[..., 0], b[..., 0]))
    iy = np.maximum(0.0, np.minimum(a[..., 3], b[..., 3]) - np.maximum(a[..., 1], b[..., 1]))
    inter = ix * iy
    aa = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    bb = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (aa + bb - inter)

def iou_shift_2d(s, d):
    '''闭式解：边长 s 的正方形，x、y 各偏 d。'''
    inter = max(0.0, s - d) ** 2
    return inter / (2.0 * s * s - inter)

def iou_shift_1d(s, d):
    '''闭式解：只在一个方向偏 d。'''
    return max(0.0, s - d) / (s + d)

# 闭式解 vs 直接用几何算 —— 两条路必须对上
for s in [8.0, 16.0, 64.0]:
    for d in [0.0, 1.0, 2.0, 3.7]:
        g = np.array([0.0, 0.0, s, s])
        p2 = np.array([d, d, s + d, s + d])
        p1 = np.array([d, 0.0, s + d, s])
        assert np.isclose(iou_xyxy(g, p2), iou_shift_2d(s, d)), (s, d)
        assert np.isclose(iou_xyxy(g, p1), iou_shift_1d(s, d)), (s, d)
print('OK 闭式解与几何计算一致（2D 对角 + 1D 单轴，各 12 组）')

# 本课最重要的两个数字
i8, i64 = iou_shift_2d(8, 2), iou_shift_2d(64, 2)
assert np.isclose(i8, 36 / 92) and np.isclose(i64, 3844 / 4348)
print()
print(f'8x8   框对角偏 2px -> IoU = {i8:.6f}   (= 36/92)')
print(f'64x64 框对角偏 2px -> IoU = {i64:.6f}   (= 3844/4348)')
print(f'**相差 {i64/i8:.2f} 倍；8px 的框已经掉到 0.5 阈值以下，会被判成负样本。**')

In [ ]:
# 完整的位移-IoU 表 + d=0 处的斜率
print('IoU(s, d)  —— 2D 对角位移')
print(f"{'边长 s':>9}" + ''.join(f'{("d=" + str(d) + "px"):>10}' for d in [1, 2, 3, 4]) + f"{'斜率 dIoU/dd|0':>16}")
for s in [8, 16, 32, 64, 128]:
    slope = (iou_shift_2d(s, 1e-7) - 1.0) / 1e-7          # 数值导数
    assert abs(slope - (-4.0 / s)) < 1e-3, (s, slope)      # 闭式解 -4/s
    print(f'{s:>7d}px' + ''.join(f'{iou_shift_2d(s, d):>10.4f}' for d in [1, 2, 3, 4]) + f'{slope:>16.5f}')

print()
print('1D 单轴位移（作对照，衰减慢一半）:')
print(f"{'边长 s':>9}" + ''.join(f'{("d=" + str(d) + "px"):>10}' for d in [1, 2, 3, 4]) + f"{'斜率':>16}")
for s in [8, 64]:
    slope1 = (iou_shift_1d(s, 1e-7) - 1.0) / 1e-7
    assert abs(slope1 - (-2.0 / s)) < 1e-3
    print(f'{s:>7d}px' + ''.join(f'{iou_shift_1d(s, d):>10.4f}' for d in [1, 2, 3, 4]) + f'{slope1:>16.5f}')

print()
print('==> **IoU 对绝对位移的敏感度 dIoU/dd|_0 = -4/s（2D）、-2/s（1D），与边长成反比。**')

# 但 IoU 本身是**尺度不变**的：把位移写成相对量 d = alpha*s，结果与 s 无关
print()
print('反面验证：把位移写成相对量 d = alpha*s，IoU 与 s 完全无关 ——')
for alpha in [0.05, 0.10, 0.25]:
    vals = [iou_shift_2d(s, alpha * s) for s in [8, 16, 32, 64, 128, 1000]]
    assert np.allclose(vals, vals[0]), vals
    print(f'  alpha={alpha:.2f}: ' + ' '.join(f'{v:.4f}' for v in vals) + '   <- 全部相同')
print()
print('**所以 IoU 惩罚的是「相对误差」，它对尺度是绝对公平的。**')
print('  不公平来自另一侧：现实中的定位误差绝大部分是**绝对像素量级**的')
print('  （标注手抖 +-1~2px、特征网格量化 stride/2、运动模糊几个像素）。')
print('  **用一把按相对误差计分的尺子，去量一堆绝对误差 —— 尺度不公平就此产生。**')

## 2 · 临界位移：把 IoU 阈值翻译成「允许偏几个像素」

由 `(s-d)^2 / (2s^2-(s-d)^2) = t` 反解：

$$d^{*}(s,t)=s\left(1-\sqrt{\frac{2t}{1+t}}\right)$$

`s` 只是一个比例因子 —— **临界位移与目标边长严格成正比**。

In [ ]:
def critical_displacement(s, tau):
    '''让 2D 对角 IoU 恰好等于 tau 的最大允许偏移。'''
    return s * (1.0 - math.sqrt(2.0 * tau / (1.0 + tau)))

# 系数验证
k50, k75 = critical_displacement(1.0, 0.5), critical_displacement(1.0, 0.75)
print(f'tau=0.50 -> d* = {k50:.6f} * s      (即 d* ~ 0.1835 * s)')
print(f'tau=0.75 -> d* = {k75:.6f} * s')
assert abs(k50 - 0.183503) < 1e-5 and abs(k75 - 0.074180) < 1e-5

# 自洽性：把 d* 代回去必须恰好得到 tau
for s in [8, 16, 32, 64]:
    for tau in [0.3, 0.5, 0.75, 0.9]:
        assert abs(iou_shift_2d(s, critical_displacement(s, tau)) - tau) < 1e-9
print('OK 反解自洽（16 组 s x tau 代回闭式解均命中阈值）')

print()
print('「IoU 阈值」翻译成「允许偏几个像素」:')
print(f"{'边长':>8}" + ''.join(f'{("tau=" + str(t)):>12}' for t in [0.5, 0.75, 0.9]))
for s in [8, 16, 32, 64, 128]:
    print(f'{s:>6d}px' + ''.join(f'{critical_displacement(s, t):>11.2f}px' for t in [0.5, 0.75, 0.9]))

d8, d64 = critical_displacement(8, 0.5), critical_displacement(64, 0.5)
assert abs(d8 - 1.4680) < 1e-3 and abs(d64 - 11.7442) < 1e-3
assert abs(d64 / d8 - 8.0) < 1e-9, '临界位移与边长严格成正比 -> 比值就是尺寸比'
print()
print(f'**同一条 IoU>=0.5 的判定线：对 8px 目标要求「{d8:.2f}px 以内」，')
print(f'  对 64px 目标要求「{d64:.2f}px 以内」—— 后者的容忍度是前者的 {d64/d8:.0f} 倍。**')
print(f'  而 AP@0.75 对 8px 目标只允许 {critical_displacement(8, 0.75):.2f}px —— **亚像素级，比标注精度还高。**')

In [ ]:
# 把真实误差源的量级并排放上去
ERROR_SOURCES = [
    ('人工标注抖动（每条边）',        2.0),
    ('特征网格量化（stride 8 中心）',  4.0),
    ('回归头固有噪声（~0.5 格）',      4.0),
    ('运动模糊 / 卷帘快门 @120km/h',   2.0),
    ('相机标定与时间戳对齐',           1.5),
]
print(f"{'误差来源':<30}{'典型量级':>10}{'8px 目标':>14}{'64px 目标':>14}")
tol8, tol64 = critical_displacement(8, 0.5), critical_displacement(64, 0.5)
for name, mag in ERROR_SOURCES:
    v8 = '致命' if mag > tol8 else ('临界' if mag > 0.6 * tol8 else '可忽略')
    v64 = '致命' if mag > tol64 else ('临界' if mag > 0.6 * tol64 else '可忽略')
    print(f'{name:<30}{mag:>9.1f}px{v8:>14}{v64:>14}')
print(f"{'—— IoU@0.5 的容忍度':<30}{'':>10}{tol8:>13.2f}px{tol64:>13.2f}px")

n_fatal_8 = sum(1 for _, m in ERROR_SOURCES if m > tol8)
n_fatal_64 = sum(1 for _, m in ERROR_SOURCES if m > tol64)
assert n_fatal_8 == 5 and n_fatal_64 == 0, (n_fatal_8, n_fatal_64)
print()
print(f'**5 个误差源里，对 8px 目标有 {n_fatal_8} 个是致命的；对 64px 目标是 {n_fatal_64} 个。**')
print()
print('推论（面试可直接讲）: COCO 的 AP@[.5:.95] 对小目标天然不利 ——')
for tau in [0.5, 0.75, 0.85, 0.95]:
    print(f'  tau={tau:.2f}: 8px 允许 {critical_displacement(8, tau):.2f}px，'
          f'64px 允许 {critical_displacement(64, tau):.2f}px')
print('  tau 一过 0.75，8px 目标的容忍度就跌破 1 个像素 -> 高阈值段的 AP 直接归零。')
print('  **AP_S 里有一块是「指标定义造成的」，与模型好坏无关。要和「模型的账」分开记。**')

## 3 · 正样本稀缺：anchor 匹配的**上界**

设 anchor 边长 `A`、目标边长 `s < A`。**即使中心完美重合、形状完美匹配**，
目标也完全落在 anchor 内部，此时

$$\mathrm{IoU}_{\max}(s,A)=\frac{s^2}{A^2}$$

这是**上界** —— 中心还没偏就已经封顶了。

In [ ]:
# RetinaNet 标准 anchor：base {32,64,128,256,512} x scales {2^0, 2^(1/3), 2^(2/3)}
SCALES = [2 ** 0, 2 ** (1 / 3), 2 ** (2 / 3)]
BASES = [32, 64, 128, 256, 512]
ANCHORS = sorted(b * sc for b in BASES for sc in SCALES)
print('最小的 6 个 anchor 边长:', [round(a, 2) for a in ANCHORS[:6]])
print('最小 anchor A_min =', ANCHORS[0], 'px')

def best_possible_iou(obj_s, anchors=ANCHORS):
    '''中心完美对齐、方形目标，能拿到的最好 IoU（上界）。'''
    return max(min(obj_s, a) ** 2 / max(obj_s, a) ** 2 for a in anchors)

print()
print(f"{'目标边长':>10}{'最好可能 IoU':>16}   0.5 阈值下能否成为正样本")
for s in [4, 8, 12, 16, 20, 22.63, 24, 32, 64]:
    b = best_possible_iou(s)
    tag = '是' if b >= 0.5 else '**否 —— 数学上不可能**'
    print(f'{s:>8.5g}px{b:>16.4f}   {tag}')

assert np.isclose(best_possible_iou(8), 0.0625)
assert np.isclose(best_possible_iou(16), 0.25)
assert best_possible_iou(22.62) < 0.5 <= best_possible_iou(22.64)
s_dead = 32 / math.sqrt(2)
print()
print(f'**8px 目标对 32px anchor 的最好可能 IoU 只有 {best_possible_iou(8):.4f} = (8/32)^2。**')
print(f'  能达到 IoU 0.5 的最小目标边长 = 32/sqrt(2) = {s_dead:.4f} px —— **这是一条死线**。')
print('  低于它的目标，在整个训练过程中一次监督信号都收不到。模型不是学不好，是从来没被教过。')

In [ ]:
# 把「中心量化」也算进去：目标中心均匀落在图上，取全图 anchor 的最大 IoU（蒙特卡洛）
def max_iou_over_grid(obj_s, levels, n=20000, rng=rng):
    '''levels: [(stride, [anchor 边长, ...]), ...]。方形目标，中心均匀分布。'''
    best = np.zeros(n)
    cx = rng.uniform(0, 256, n); cy = rng.uniform(0, 256, n)
    for stride, sizes in levels:
        ax0 = (np.floor(cx / stride) + 0.5) * stride     # 所在格的 anchor 中心
        ay0 = (np.floor(cy / stride) + 0.5) * stride
        for ox in (-1, 0, 1):                            # 搜 3x3 邻域，取最好
            for oy in (-1, 0, 1):
                dx = np.abs(cx - (ax0 + ox * stride))
                dy = np.abs(cy - (ay0 + oy * stride))
                for A in sizes:
                    iw = np.clip((obj_s + A) / 2 - dx, 0, min(obj_s, A))
                    ih = np.clip((obj_s + A) / 2 - dy, 0, min(obj_s, A))
                    inter = iw * ih
                    best = np.maximum(best, inter / (obj_s ** 2 + A * A - inter))
    return best

RETINA = [(8, [32 * t for t in SCALES]), (16, [64 * t for t in SCALES]),
          (32, [128 * t for t in SCALES]), (64, [256 * t for t in SCALES]),
          (128, [512 * t for t in SCALES])]
WITH_P2 = [(4, [8 * t for t in SCALES]), (8, [16 * t for t in SCALES]),
           (16, [32 * t for t in SCALES]), (32, [64 * t for t in SCALES])]

print(f"{'目标':>7}{'RetinaNet P(>=0.5)':>21}{'平均 maxIoU':>14}"
      f"{'加P2+小anchor P(>=0.5)':>26}{'平均':>9}")
rows = {}
for s in [8, 16, 24, 32, 64]:
    a = max_iou_over_grid(s, RETINA)
    b = max_iou_over_grid(s, WITH_P2)
    rows[s] = (a, b)
    print(f'{s:>5d}px{np.mean(a >= 0.5):>21.3f}{a.mean():>14.3f}'
          f'{np.mean(b >= 0.5):>26.3f}{b.mean():>9.3f}')

assert np.mean(rows[8][0] >= 0.5) == 0.0, 'RetinaNet 配置下 8px 目标匹配率必须是 0'
assert np.mean(rows[16][0] >= 0.5) == 0.0, '16px 同样是 0'
assert np.mean(rows[32][0] >= 0.5) == 1.0
assert np.mean(rows[8][1] >= 0.5) > 0.85, '把 anchor 缩到 8 并加 P2 后应当大幅回升'
print()
print('**RetinaNet 的标准配置下，8px 与 16px 目标的正样本匹配率精确地是 0.000。**')
print(f'  把 anchor 基准缩到 8 并加上 P2（stride 4）后，8px 目标的匹配率回到 '
      f'{np.mean(rows[8][1] >= 0.5):.1%}。')
print()
print('==> 检查清单里最该加的一行（一次除法就能算）:')
print('    (最小目标尺寸 / 最小 anchor 边长)^2  <  分配阈值 ?  -> 是，则该尺寸永远拿不到正样本')
print('    修法三层: (1) 重新聚类 anchor / 换宽高比判据  (2) ATSS / SimOTA 自适应')
print('             (3) 彻底放弃 IoU 分配（FCOS 的 center sampling、RFLA 的感受野高斯）')
print('    **注意别用「全局把阈值从 0.5 降到 0.3」—— 那会让大目标吸进一堆劣质正样本。**')

## 4 · stride 与特征预算：8 像素在 stride 32 上只有 0.25 个格子

边长 `s` 的目标在 stride 为 `r` 的特征层上占 `s/r` 个格子（线性）、`(s/r)^2` 个格子（面积）。
纯除法，但结论比想象中狠。

In [ ]:
print('特征格子预算')
print(f"{'层':>4}{'stride':>8}{'8px(线性/面积)':>20}{'64px(线性/面积)':>22}{'格子数之比':>12}")
for lvl, r in [('P2', 4), ('P3', 8), ('P4', 16), ('P5', 32), ('P6', 64)]:
    a_lin, b_lin = 8 / r, 64 / r
    ratio = (b_lin ** 2) / (a_lin ** 2)
    print(f'{lvl:>4}{r:>8d}'
          f'{f"{a_lin:.3g} / {a_lin**2:.4g}":>20}'
          f'{f"{b_lin:.3g} / {b_lin**2:.4g}":>22}'
          f'{ratio:>11.0f}x')

assert np.isclose(8 / 32, 0.25) and np.isclose((8 / 32) ** 2, 0.0625)
print()
print(f'**stride 32 的特征图上，8 像素的目标只占 {8/32:.2f} 个格子（面积 {(8/32)**2:.4f} 格）——')
print('  它连一个格子都填不满，必须和周围 16 倍面积的背景共享同一个特征向量。**')

# 输入侧的像素预算
print()
for s in [8, 16, 32, 64]:
    print(f'  {s:>3d}x{s:<3d} RGB patch = {s*s*3:>6d} 个数')
assert (64 * 64 * 3) / (8 * 8 * 3) == 64.0
print(f'  ==> 64px 目标的原始像素预算是 8px 目标的 {(64*64*3)//(8*8*3)} 倍。')

# 「2x2 格子」经验线 -> 最细 stride 决定了可靠工作的最小目标
print()
print('经验线：目标在最细特征层上至少要有 2x2 个格子，检测头才能同时表达「有没有」与「在哪多大」')
for finest in [4, 8, 16]:
    print(f'  最细 stride {finest:>2d}  ->  可靠工作的最小目标约 {2*finest:>3d} px')

# 加 P2 的代价：neck/head 的空间格子总数
base = sum(1.0 / (r ** 2) for r in [8, 16, 32, 64, 128])       # P3..P7
withp2 = base + 1.0 / (4 ** 2)                                  # P2..P7
print()
print(f'加 P2 的代价:  P3..P7 = {base:.5f}*HW 格   ->   P2..P7 = {withp2:.5f}*HW 格')
print(f'  ==> **约 {withp2/base:.2f} 倍**。这就是「加 P2」这个最直接解法的价目表。')
assert 3.95 < withp2 / base < 4.05
print('  而且这 4 倍开销的收益**完全集中在最小的那个尺寸桶**（P2 上几乎不分配中大目标），')
print('  显存与 NMS 前的候选框数也同步上升 —— 车端往往不可接受。')

In [ ]:
# 特征混叠：两个小目标落进同一个格子的概率（1D，中心均匀分布）
def same_cell_prob(gap, stride):
    return max(0.0, 1.0 - gap / stride)

print('两个目标中心相距 gap 时，落进同一个特征格的概率')
print(f"{'gap':>7}" + ''.join(f'{("stride " + str(r)):>13}' for r in [8, 16, 32]))
for gap in [10, 20, 30, 50]:
    print(f'{gap:>5d}px' + ''.join(f'{same_cell_prob(gap, r):>13.3f}' for r in [8, 16, 32]))

assert np.isclose(same_cell_prob(10, 16), 0.375)
assert np.isclose(same_cell_prob(20, 32), 0.375)
assert same_cell_prob(50, 32) == 0.0

# 蒙特卡洛校验闭式解
n = 200000
cx = rng.uniform(0, 320, n)
for gap, r in [(10, 16), (20, 32), (30, 32)]:
    same = (np.floor(cx / r) == np.floor((cx + gap) / r)).mean()
    assert abs(same - same_cell_prob(gap, r)) < 0.01, (gap, r, same)
print()
print('OK 闭式解 max(0, 1-gap/stride) 与蒙特卡洛一致')

# center-based 头的冲突消歧（FCOS：同格冲突时取面积最小的 GT）
print()
print('同格冲突时 center-based 头会发生什么（FCOS 用「取面积最小的 GT」消歧）:')
gts = [('主限速牌 60', 18.0), ('下挂辅助牌 货车', 12.0)]   # (名字, 边长)
winner = min(gts, key=lambda t: t[1] ** 2)
loser = max(gts, key=lambda t: t[1] ** 2)
print(f'  两块牌中心相距 10px，stride 16 -> 同格概率 {same_cell_prob(10, 16):.1%}')
print(f'  同格时只有「{winner[0]}」能被这个特征点认领，「{loser[0]}」**直接漏掉**')
assert winner[0] == '下挂辅助牌 货车'
print()
print('==> 后果是**召回天花板**而不是「置信度低」—— 调 score 阈值救不回来。')
print('    诊断口诀：**调阈值救不回来的漏检，先怀疑特征混叠。**')
print('    TSR 里的高发场景：门架上并排的指路牌、限速牌下挂的辅助牌、连续摆放的施工锥牌。')

## 5 · 有效感受野：理论感受野是一张空头支票

`n` 层 stride-1 的 3x3 卷积，理论感受野半径 `R = n`（每层向外扩 1）。
但把 3x3 近似成 3 抽头均匀核，单层 1D 方差 `((-1)^2+0^2+1^2)/3 = 2/3`，
`n` 层方差可加（中心极限定理）：

$$\sigma_{\mathrm{ERF}}=\sqrt{\frac{2n}{3}},\qquad
\frac{\sigma_{\mathrm{ERF}}}{R_{\mathrm{TRF}}}=\sqrt{\frac{2}{3n}}\ \to\ 0$$

**理论感受野按 O(n) 增长，有效感受野只按 O(sqrt(n)) 增长。**

In [ ]:
def erf_profile(n_layers, half=400):
    '''把一个 delta 连续通过 n 层 3 抽头均匀核，得到 1D 的有效感受野权重分布。'''
    x = np.zeros(2 * half + 1); x[half] = 1.0
    k = np.ones(3) / 3.0
    for _ in range(n_layers):
        x = np.convolve(x, k, mode='same')
    return x

print(f"{'层数 n':>7}{'理论RF半径':>12}{'ERF std(实测)':>15}{'公式 sqrt(2n/3)':>17}"
      f"{'std/RF':>9}{'95%质量半径':>13}{'占RF比例':>10}")
for n_layers in [5, 10, 20, 40, 80]:
    p = erf_profile(n_layers)
    half = (len(p) - 1) // 2
    idx = np.arange(len(p)) - half
    var = (p * idx ** 2).sum() / p.sum()
    std = math.sqrt(var)
    formula = math.sqrt(2 * n_layers / 3)
    r95 = 1.96 * std
    assert abs(std - formula) < 1e-6, (n_layers, std, formula)     # 精确命中闭式解
    print(f'{n_layers:>7d}{n_layers:>12d}{std:>15.3f}{formula:>17.3f}'
          f'{std/n_layers:>9.3f}{r95:>13.2f}{r95/n_layers:>9.1%}')

p20 = erf_profile(20); h20 = (len(p20) - 1) // 2
idx20 = np.arange(len(p20)) - h20
std20 = math.sqrt((p20 * idx20 ** 2).sum() / p20.sum())
assert abs(std20 - 3.651) < 1e-3
assert 1.96 * std20 / 20 < 0.40, 'n=20 时 95% 权重只落在理论感受野不到 40% 的半径内'
print()
print(f'**n=20 时：理论感受野半径 20，但 95% 的权重只落在半径 {1.96*std20:.2f} 之内'
      f'（理论值的 {1.96*std20/20:.1%}）。**')
print('  网络越深，这张支票就越空 —— TRF 按 O(n) 涨，ERF 只按 O(sqrt(n)) 涨。')
print()
print('注：真实网络还有 ReLU/BN/跳连，ERF 会更集中；且训练后的 ERF 通常比初始化时更大。')
print('    所以「用初始化模型测 ERF」会低估真实值 —— 要在**训练好的**模型上用梯度回传法测。')

In [ ]:
# 目标在 ERF 里的「信噪比」：目标本身占该神经元有效输入的多少
def frac_in_erf_1d(obj_px, sigma_px):
    '''高斯 ERF 下，宽度 obj_px 的目标覆盖的权重比例（1D）。'''
    return math.erf(obj_px / (2.0 * sigma_px * math.sqrt(2.0)))

SIGMA = 30.0     # 某个主干中层的 ERF 在输入图像上的标准差（典型量级）
print(f'设该特征位置的 ERF 在输入上 sigma = {SIGMA:.0f} px')
print(f"{'目标边长':>10}{'占ERF(1D)':>12}{'占ERF(2D)':>12}   解读")
vals = {}
for s in [8, 16, 32, 64, 128]:
    f1 = frac_in_erf_1d(s, SIGMA); f2 = f1 * f1
    vals[s] = f2
    note = ('该神经元 99% 的有效输入是背景' if f2 < 0.02 else
            '仍被上下文淹没' if f2 < 0.10 else
            '开始成为主要信号源' if f2 < 0.40 else
            '目标本身占了一半以上' if f2 < 0.90 else '几乎纯信号')
    print(f'{s:>8d}px{f1:>12.4f}{f2:>12.4f}   {note}')

assert abs(frac_in_erf_1d(8, SIGMA) - 0.1061) < 1e-3
assert abs(frac_in_erf_1d(64, SIGMA) - 0.7139) < 1e-3
assert vals[64] / vals[8] > 40, '2D 上 64px 与 8px 的信噪比应差 40 倍以上'
print()
print(f'**同一个神经元，看 64px 目标时 {vals[64]:.1%} 的有效输入来自目标本身，')
print(f'  看 8px 目标时只有 {vals[8]:.2%} —— 信号被上下文按 {vals[64]/vals[8]:.0f} 倍的比例稀释。**')
print()
print('==> 所以「小目标需要更大感受野」是**反的**：问题不是感受野太小，是感受野相对目标太大。')
print('    继续加深、继续堆大核 -> ERF 更大 -> 稀释更严重。')
print('    小目标需要的是**与目标尺寸匹配**的感受野 —— 这正是 FPN 层级分配的第一性原理')
print('    （k = k0 + log2(sqrt(wh)/224) 就是在做这件匹配）。')
print('    反面例子：RTMDet 的 5x5 大核、DETR 的全局 attention 都在**放大** ERF，')
print('    那是为大目标与长程上下文服务的设计，**「大核对小目标好」是必须警惕的误传**。')

## 6 · 标注噪声：小目标的标签本身就带噪

前五节都在讲模型侧。这一节在**数据侧**，而且它设定的是**指标的上限** ——
无论模型多强都突破不了。

保守假设：标注员画框时每条边独立有 `U(-2, +2)` px 误差。
**对 8px 框这是 25% 的相对误差；对 64px 框只有 3.1%。**

In [ ]:
def annotation_iou_samples(s, jitter=2.0, n=100000, kind='uniform', rng=rng):
    '''真值框 vs 带噪标注框的 IoU 分布。'''
    gt = np.tile(np.array([0.0, 0.0, s, s]), (n, 1))
    e = (rng.uniform(-jitter, jitter, size=(n, 4)) if kind == 'uniform'
         else rng.normal(0.0, jitter, size=(n, 4)))
    nb = gt + e
    nb[:, 2] = np.maximum(nb[:, 2], nb[:, 0] + 0.5)      # 防止退化成负宽高
    nb[:, 3] = np.maximum(nb[:, 3], nb[:, 1] + 0.5)
    return iou_xyxy(gt, nb)

print('每条边 U(-2, +2) px 抖动下，标注 vs 真值的 IoU')
print(f"{'边长':>7}{'相对误差':>10}{'平均IoU':>10}{'P(>=0.5)':>11}{'P(>=0.75)':>12}{'5%分位':>9}")
res = {}
for s in [8, 16, 32, 64, 128]:
    v = annotation_iou_samples(s); res[s] = v
    print(f'{s:>5d}px{2/s:>10.1%}{v.mean():>10.4f}{np.mean(v>=0.5):>11.4f}'
          f'{np.mean(v>=0.75):>12.4f}{np.percentile(v,5):>9.4f}')

assert 0.60 < res[8].mean() < 0.65, res[8].mean()
assert np.mean(res[8] >= 0.75) < 0.15
assert np.mean(res[64] >= 0.75) > 0.99
assert abs(2 / 8 - 0.25) < 1e-12, '+-2px 对 8px 框正好是 25% 的相对误差'
print()
print(f'**+-2px 对 8px 框是 {2/8:.0%} 的相对误差；对 64px 框只有 {2/64:.1%}。**')

# 这就是「完美模型」的 AP 上限：模型输出等于物理真值，但标签是带噪的
print()
print('推论：一个「完美模型」（输出恰好等于物理真值）在各尺寸桶上的 AP 上限')
print(f"{'边长':>7}{'AP@0.5 上限':>14}{'AP@0.75 上限':>15}")
for s in [8, 16, 32, 64]:
    print(f'{s:>5d}px{np.mean(res[s]>=0.5):>14.1%}{np.mean(res[s]>=0.75):>15.1%}')
print()
print(f'**8px 桶上 AP@0.75 的天花板只有 {np.mean(res[8]>=0.75):.1%} —— 这个指标测的是标注员，不是模型。**')

# 抖动模型换成高斯，结论不变
vg = annotation_iou_samples(8, jitter=1.0, kind='normal')
assert 0.65 < vg.mean() < 0.72, vg.mean()
print(f'  换成每边 N(0,1) 的更温和抖动：8px 框平均 IoU 仍只有 {vg.mean():.3f}，'
      f'达到 0.75 的比例 {np.mean(vg>=0.75):.1%}')
print()
print('三条结论:')
print(' 1) AP@0.75 在小目标桶上测的是标注员 -> 小目标只报 AP@0.5 + 召回')
print(' 2) **训练信号本身带噪** -> 小目标回归分支很大程度在学噪声（框抖、时序平滑收益大）')
print(' 3) 与前四个原因**完全正交** -> 加 P2 / 换 NWD / 提分辨率都改善不了标注质量')
print()
print('工程动作：抽 200 个小目标让第二位标注员盲标，算两份标注的 IoU 分布 ——')
print('**那个分布就是你在该尺寸桶上的指标上限。** 不做这个测量就定 OKR，很可能在追一个不可能的数。')

In [ ]:
# 可分性 d'：在**固定的镜头/传感器模糊**下，「60」与「80」随尺寸还能不能分开
FONT5x7 = {
    '0': ['01110', '10001', '10011', '10101', '11001', '10001', '01110'],
    '3': ['11111', '00010', '00100', '00010', '00001', '10001', '01110'],
    '6': ['00110', '01000', '10000', '11110', '10001', '10001', '01110'],
    '8': ['01110', '10001', '10001', '01110', '10001', '10001', '01110'],
}

def render_sign(text, R=96):
    '''高分辨率渲染一块圆形限速牌：白底 + 暗环（红环的灰度替身）+ 黑数字。'''
    img = np.ones((R, R))
    yy, xx = np.mgrid[0:R, 0:R]
    rad = np.sqrt((xx - (R - 1) / 2) ** 2 + (yy - (R - 1) / 2) ** 2) / (R / 2)
    img[rad > 1.0] = 0.35                       # 牌外背景
    img[(rad <= 1.0) & (rad > 0.80)] = 0.25     # 环，厚度 = 直径的 10%
    gw, gh = int(0.20 * R), int(0.40 * R)       # 每个数字：宽 0.20R，高 0.40R
    for k, ch in enumerate(text):
        bm = FONT5x7[ch]
        x0 = int(R / 2 - 0.24 * R + k * 0.24 * R); y0 = int(R / 2 - 0.20 * R)
        for i in range(gh):
            for j in range(gw):
                if bm[i * 7 // gh][j * 5 // gw] == '1' and 0 <= y0 + i < R and 0 <= x0 + j < R:
                    img[y0 + i, x0 + j] = 0.0
    return img

def area_downsample(img, D):
    R = img.shape[0]; e = np.linspace(0, R, D + 1).astype(int); out = np.zeros((D, D))
    for i in range(D):
        for j in range(D):
            out[i, j] = img[e[i]:e[i + 1], e[j]:e[j + 1]].mean()
    return out

def blur(img, sig):
    '''可分离高斯模糊。sig 的单位是**成像后的像素** —— 镜头/ISP 的模糊是绝对量级的。'''
    r = max(1, int(3 * sig)); x = np.arange(-r, r + 1)
    k = np.exp(-x ** 2 / (2 * sig * sig)); k /= k.sum()
    out = np.apply_along_axis(lambda m: np.convolve(m, k, mode='same'), 0, img)
    return np.apply_along_axis(lambda m: np.convolve(m, k, mode='same'), 1, out)

sign60, sign80 = render_sign('60'), render_sign('80')
SENSOR_NOISE = 4 / 255.0        # 典型车载相机的读出噪声量级
BLUR_PX = 0.8                   # 镜头 + ISP 的模糊，单位是像素（**与目标大小无关**）

print(f'固定 {BLUR_PX} px 的镜头模糊 + sigma={SENSOR_NOISE:.4f} 的传感器噪声')
print(f"{'牌子直径 D':>12}{'笔画宽 D/8':>12}{'可分性 d(60 vs 80)':>22}")
dp = {}
for D in [4, 6, 8, 12, 16, 24, 32, 48, 64]:
    a = blur(area_downsample(sign60, D), BLUR_PX)
    b = blur(area_downsample(sign80, D), BLUR_PX)
    dp[D] = float(np.linalg.norm(a - b) / SENSOR_NOISE)
    print(f'{D:>10d}px{D/8:>12.2f}{dp[D]:>22.2f}')

ks = [4, 6, 8, 12, 16, 24, 32, 48, 64]
assert all(dp[x] < dp[y] for x, y in zip(ks[:-1], ks[1:])), '可分性必须随尺寸单调上升'
assert dp[32] > 5 * dp[8], (dp[8], dp[32])
expo = math.log(dp[32] / dp[8]) / math.log(32 / 8)
assert expo > 1.4, expo
print()
print(f'**尺寸从 8px 到 32px（4 倍），可分性 d 从 {dp[8]:.1f} 涨到 {dp[32]:.1f}'
      f'（{dp[32]/dp[8]:.1f} 倍）—— 局部指数约 D^{expo:.2f}，比线性还快。**')
print('  原因：模糊是**绝对像素量级**的，它抹掉的细节量固定，')
print('        但这个固定量对小目标是致命比例、对大目标可忽略 —— 又一个「绝对 vs 相对」。')
print()
print('奈奎斯特判据（限速牌的数字笔画宽约为直径的 1/8，环厚约 1/10）:')
print(f'  要读出数字，笔画需 >= 2px -> D >= {8*2} px')
print(f'  要检出牌子，环需 >= 2px   -> D >= {10*2} px')
f60 = (1920 / 2) / math.tan(math.radians(30))
print(f'  1920p/60deg 相机、0.6m 的牌: 16px 对应 {f60*0.6/16:.1f} m，20px 对应 {f60*0.6/20:.1f} m')
print('  **这是物理极限，不是模型能力问题** —— 唯一的解法是给更多像素。')

## ✏️ 练习 1：构造一个「尺度公平」的 IoU 阈值

现在的问题是：**同一个 IoU 阈值对不同尺度不公平**。反过来想 ——
如果我们规定所有尺度都享有**同一个绝对位移容忍度** `d_tol`（像素），
那么每个尺寸对应的 IoU 阈值应该是多少？

实现 `scale_fair_threshold(s, d_tol)`：返回边长 `s` 的目标在容忍 `d_tol` 像素
对角位移时对应的 IoU 阈值。

这正是模块 03「尺度自适应阈值」的构造方法 —— **一行代码，但它把不公平消掉了。**

In [ ]:
def scale_fair_threshold(s, d_tol):
    # TODO: 返回边长 s、容忍 d_tol 像素对角位移时等价的 IoU 阈值
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(scale_fair_threshold(8, 2) - 36 / 92) < 1e-9
assert abs(scale_fair_threshold(64, 2) - 3844 / 4348) < 1e-9

# 单调性：同样容忍 2px，目标越大要求的阈值越高
taus = [scale_fair_threshold(s, 2) for s in [8, 16, 32, 64, 128, 512]]
assert all(a < b for a, b in zip(taus[:-1], taus[1:])), taus
assert taus[-1] > 0.98, '目标足够大时阈值趋近 1'

# **核心自洽性**：用这套阈值反推临界位移，所有尺度必须得到同一个 d_tol
for d_tol in [1.0, 2.0, 3.5]:
    for s in [8, 16, 32, 64, 128]:
        back = critical_displacement(s, scale_fair_threshold(s, d_tol))
        assert abs(back - d_tol) < 1e-8, (s, d_tol, back)
print('OK 自洽：这套阈值下，所有尺度的绝对位移容忍度**完全相同**')

print()
print(f"{'边长':>7}{'固定阈值 0.5':>14}{'尺度公平阈值(d_tol=2px)':>26}")
for s in [8, 16, 32, 64, 128]:
    print(f'{s:>5d}px{0.5:>14.4f}{scale_fair_threshold(s, 2):>26.4f}')
print()
print('对比：固定 0.5 阈值下，8px 目标只容忍 1.47px、64px 容忍 11.74px（差 8 倍）；')
print('      尺度公平阈值下，所有尺度都容忍 2.00px。')
print('✅ 练习 1 通过：**不公平不在 IoU 公式里，在「所有尺度共用一个阈值」这个约定里。**')

## ✏️ 练习 2：anchor 覆盖诊断器

实现 `anchor_coverage(anchor_sizes, obj_sizes, thr=0.5)`，返回

```python
{'best_iou': [...],            # 每个目标的最好可能 IoU（上界）
 'impossible': [...],          # 布尔列表：该尺寸是否「数学上不可能」成为正样本
 'impossible_frac': float,     # 不可能的比例
 'min_anchor_needed': float}   # 要让**最小**的目标也能达到 thr，最小 anchor 应该 <= 多少
```

提示：`IoU_max(s, A) = min(s,A)^2 / max(s,A)^2`；要让边长 `s_min` 的目标达到 `thr`，
需要 `A <= s_min / sqrt(thr)`。

In [ ]:
def anchor_coverage(anchor_sizes, obj_sizes, thr=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r = anchor_coverage(ANCHORS, [8, 16, 24, 32], thr=0.5)
assert abs(r['best_iou'][0] - 0.0625) < 1e-12, r['best_iou'][0]
assert abs(r['best_iou'][1] - 0.25) < 1e-12
assert r['impossible'] == [True, True, False, False], r['impossible']
assert abs(r['impossible_frac'] - 0.5) < 1e-12
assert abs(r['min_anchor_needed'] - 8 / math.sqrt(0.5)) < 1e-9, r['min_anchor_needed']

# 用求出来的 min_anchor_needed 作为最小 anchor 重算 -> 应当全部可行
fixed = sorted([r['min_anchor_needed'] * t for t in SCALES] + ANCHORS)
r2 = anchor_coverage(fixed, [8, 16, 24, 32], thr=0.5)
assert r2['impossible_frac'] == 0.0, r2['impossible']

# 阈值更严时，需要的 anchor 更小
assert (anchor_coverage(ANCHORS, [8], thr=0.75)['min_anchor_needed']
        < anchor_coverage(ANCHORS, [8], thr=0.5)['min_anchor_needed'])

print(f"{'目标':>7}{'最好可能IoU':>14}{'是否不可能':>12}")
for s, b, imp in zip([8, 16, 24, 32], r['best_iou'], r['impossible']):
    print(f'{s:>5d}px{b:>14.4f}{("是" if imp else "否"):>12}')
print()
print(f"当前配置下 {r['impossible_frac']:.0%} 的尺寸档「数学上不可能」成为正样本")
print(f"要救回 8px 目标，最小 anchor 必须 <= {r['min_anchor_needed']:.2f} px"
      f"（当前是 {min(ANCHORS):.0f} px）")
print('✅ 练习 2 通过：**这一行除法应该出现在每个检测项目的启动检查清单里。**')

## ✏️ 练习 3：标注噪声决定的 AP 上限

实现 `ap_ceiling(s, jitter=2.0, tau=0.5, n=50000)`：
返回「一个完美模型（输出恰好等于物理真值）」在该尺寸桶上能达到的 **AP 上限** ——
也就是「带噪标注框与真值框的 IoU ≥ tau」的比例。

（可以直接复用上面的 `annotation_iou_samples`。）

In [ ]:
def ap_ceiling(s, jitter=2.0, tau=0.5, n=50000):
    # TODO: 返回 0~1 的上限
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
c8_75 = ap_ceiling(8, 2.0, 0.75)
c8_50 = ap_ceiling(8, 2.0, 0.50)
c64_75 = ap_ceiling(64, 2.0, 0.75)
assert c8_75 < 0.15, c8_75
assert c8_50 > 0.85, c8_50
assert c64_75 > 0.99, c64_75
assert 0.0 <= c8_75 <= 1.0

# 单调性：目标越大上限越高；阈值越严上限越低；抖动越大上限越低
seq = [ap_ceiling(s, 2.0, 0.75) for s in [8, 16, 32, 64]]
assert all(a <= b + 1e-9 for a, b in zip(seq[:-1], seq[1:])), seq
assert ap_ceiling(16, 2.0, 0.9) < ap_ceiling(16, 2.0, 0.5)
assert ap_ceiling(16, 4.0, 0.75) < ap_ceiling(16, 1.0, 0.75)

print(f"{'边长':>7}{'AP@0.5 上限':>14}{'AP@0.75 上限':>15}{'AP@0.9 上限':>14}")
for s in [8, 16, 32, 64, 128]:
    print(f'{s:>5d}px{ap_ceiling(s,2.0,0.5):>14.1%}'
          f'{ap_ceiling(s,2.0,0.75):>15.1%}{ap_ceiling(s,2.0,0.9):>14.1%}')
print()
print('**表里凡是明显低于 100% 的格子，那个指标测的都是标注员而不是模型。**')
print('✅ 练习 3 通过：定 OKR 之前先把这张表算出来 —— 别去追一个不可能的数')

## ✏️ 练习 4：可检测尺寸下界与最远作用距离（本模块的收官题）

把物理层（奈奎斯特）与架构层（stride）两个下界合起来，求出系统真正的作用距离。

实现 `min_size_and_range(width_px, hfov_deg, size_m, finest_stride,
stroke_ratio=1/8, nyquist_px=2.0, cells_needed=2.0)`，返回

```python
{'min_px_nyquist': …,   # 物理下界：stroke = D*stroke_ratio >= nyquist_px
 'min_px_stride':  …,   # 架构下界：D >= cells_needed * finest_stride
 'min_px':         …,   # 两者取大 —— **谁大谁是瓶颈**
 'bottleneck':     'physics' 或 'architecture',
 'max_range_m':    …}   # f*S/min_px
```

**做完之后请特别留意 `bottleneck` 这个字段随 `finest_stride` 的变化 ——
它演示了「修好一个原因之后，瓶颈会立刻换到下一个原因」。**

In [ ]:
def min_size_and_range(width_px, hfov_deg, size_m, finest_stride,
                       stroke_ratio=1/8, nyquist_px=2.0, cells_needed=2.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
r8 = min_size_and_range(1920, 60.0, 0.6, finest_stride=8)
assert abs(r8['min_px_nyquist'] - 16.0) < 1e-9, r8['min_px_nyquist']
assert abs(r8['min_px_stride'] - 16.0) < 1e-9
assert abs(r8['min_px'] - 16.0) < 1e-9
assert abs(r8['max_range_m'] - 62.35) < 0.1, r8['max_range_m']

r16 = min_size_and_range(1920, 60.0, 0.6, finest_stride=16)
assert abs(r16['min_px'] - 32.0) < 1e-9 and r16['bottleneck'] == 'architecture'
assert abs(r16['max_range_m'] - 31.18) < 0.1

r4 = min_size_and_range(1920, 60.0, 0.6, finest_stride=4)      # 加 P2
assert abs(r4['min_px'] - 16.0) < 1e-9, '加 P2 后架构下界降到 8px，物理下界 16px 成为瓶颈'
assert r4['bottleneck'] == 'physics'
assert abs(r4['max_range_m'] - r8['max_range_m']) < 1e-6, '**加 P2 一米都没多看到**'

r4k = min_size_and_range(3840, 60.0, 0.6, finest_stride=4)      # 加 P2 + 换 4K
assert r4k['max_range_m'] > 1.9 * r4['max_range_m']

print(f"{'配置':<28}{'物理下界':>10}{'架构下界':>10}{'实际下界':>10}{'瓶颈':>14}{'最远距离':>11}")
for label, kw in [('1080p, stride 16', dict(width_px=1920, finest_stride=16)),
                  ('1080p, stride 8 (基线)', dict(width_px=1920, finest_stride=8)),
                  ('1080p, stride 4 (加 P2)', dict(width_px=1920, finest_stride=4)),
                  ('4K,    stride 4 (P2+4K)', dict(width_px=3840, finest_stride=4))]:
    v = min_size_and_range(hfov_deg=60.0, size_m=0.6, **kw)
    print(f"{label:<28}{v['min_px_nyquist']:>9.1f}px{v['min_px_stride']:>9.1f}px"
          f"{v['min_px']:>9.1f}px{v['bottleneck']:>14}{v['max_range_m']:>10.1f}m")
print()
print('**注意第 2 行到第 3 行：加了 P2（代价 4 倍算力），最远距离一米都没多。**')
print('  因为瓶颈已经从架构层（stride）换到了物理层（奈奎斯特）——')
print('  此时该做的是换相机/提分辨率（第 4 行），而不是继续在架构上加码。')
print('✅ 练习 4 通过：**这就是「五个原因互相正交」在工程上的真实样子。**')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def scale_fair_threshold(s, d_tol):
    '''容忍 d_tol 像素对角位移时，边长 s 对应的等价 IoU 阈值。'''
    inter = max(0.0, s - d_tol) ** 2
    return inter / (2.0 * s * s - inter)          # 就是 iou_shift_2d(s, d_tol)

In [ ]:
# 练习 2 参考答案
def anchor_coverage(anchor_sizes, obj_sizes, thr=0.5):
    best = [max(min(s, a) ** 2 / max(s, a) ** 2 for a in anchor_sizes) for s in obj_sizes]
    imp = [b < thr for b in best]
    return {
        'best_iou': best,
        'impossible': imp,
        'impossible_frac': sum(imp) / len(imp),
        # 要让最小的目标也达到 thr: (s_min/A)^2 >= thr  =>  A <= s_min/sqrt(thr)
        'min_anchor_needed': min(obj_sizes) / math.sqrt(thr),
    }

In [ ]:
# 练习 3 参考答案
def ap_ceiling(s, jitter=2.0, tau=0.5, n=50000):
    v = annotation_iou_samples(s, jitter=jitter, n=n)
    return float(np.mean(v >= tau))

In [ ]:
# 练习 4 参考答案
def min_size_and_range(width_px, hfov_deg, size_m, finest_stride,
                       stroke_ratio=1/8, nyquist_px=2.0, cells_needed=2.0):
    f = (width_px / 2.0) / math.tan(math.radians(hfov_deg / 2.0))
    min_px_nyquist = nyquist_px / stroke_ratio          # 笔画 = D*ratio >= nyquist
    min_px_stride = cells_needed * finest_stride        # 至少 cells_needed 个格子
    min_px = max(min_px_nyquist, min_px_stride)
    return {
        'min_px_nyquist': min_px_nyquist,
        'min_px_stride': min_px_stride,
        'min_px': min_px,
        'bottleneck': 'physics' if min_px_nyquist >= min_px_stride else 'architecture',
        'max_range_m': f * size_m / min_px,
    }

---
## 🧪 真实工程胶囊：小目标专项的「五笔账」诊断脚本

可原样复制到项目里。**任何「小目标不行」的工单，先把这五笔账跑一遍再讨论方案。**

In [ ]:
RECIPE = r'''
# ── 小目标五因诊断（开工前必跑）─────────────────────────────────────────
import math, numpy as np

CFG = dict(
    # A 物理层
    width_px=1920, hfov_deg=60.0, target_m=0.60, required_range_m=60.0,
    # B 度量 / 分配层
    finest_stride=8, min_anchor=32, assign_thr=0.5,
    # C 数据层（**必须实测，不要拍脑袋**：抽 200 个小目标让第二人盲标）
    anno_jitter_px=2.0,
)

f = (CFG["width_px"]/2) / math.tan(math.radians(CFG["hfov_deg"]/2))
s = f * CFG["target_m"] / CFG["required_range_m"]           # 该距离上的像素边长

# ① 信息量：奈奎斯特下界（笔画宽约为直径 1/8，环厚约 1/10）
min_px_cls, min_px_det = 2/(1/8), 2/(1/10)                  # 16 px / 20 px
r1 = "PASS" if s >= min_px_cls else "FAIL(像素预算不够，改模型没用 -> 换相机/提分辨率/ROI)"

# ② 度量：该尺寸在 assign_thr 下的绝对位移容忍度
d_tol = s * (1 - math.sqrt(2*CFG["assign_thr"]/(1+CFG["assign_thr"])))
r2 = "PASS" if d_tol >= 3.0 else "FAIL(容忍度低于回归噪声 -> 尺度自适应阈值 / NWD)"

# ③ 分配：**最重要的一行除法**
best_iou = 1.0 if s >= CFG["min_anchor"] else (s/CFG["min_anchor"])**2
r3 = "PASS" if best_iou >= CFG["assign_thr"] else "FAIL(一个正样本都拿不到 -> 先修这个，几乎免费)"
min_anchor_needed = s / math.sqrt(CFG["assign_thr"])

# ④ 架构：特征格子数（经验线 >= 2x2）
cells = s / CFG["finest_stride"]
r4 = "PASS" if cells >= 2.0 else "FAIL(格子不够 -> 加 P2(代价x4) 或 ROI 精检)"

# ⑤ 数据：标注相对误差与指标上限
rel_err = CFG["anno_jitter_px"] / s
r5 = "PASS" if rel_err <= 0.15 else "FAIL(标签带噪 -> 小目标只报 AP@0.5+召回，别报 AP@0.75)"

for tag, val, verdict in [
    ("① 信息量  像素边长",      f"{s:.1f}px (下界 {min_px_cls:.0f})", r1),
    ("② 度量    位移容忍度",    f"{d_tol:.2f}px",                     r2),
    ("③ 分配    最好可能 IoU",  f"{best_iou:.4f} (需 {CFG['assign_thr']})", r3),
    ("④ 架构    特征格子数",    f"{cells:.2f}x{cells:.2f}",           r4),
    ("⑤ 数据    标注相对误差",  f"{rel_err:.1%}",                     r5),
]:
    print(f"{tag:<22}{val:<22}{verdict}")
print(f"
若 ③ FAIL: 最小 anchor 必须 <= {min_anchor_needed:.1f} px（当前 {CFG['min_anchor']}）")

# ── 动手顺序（严格按此，别跳步）────────────────────────────────────────
# 1) ③ 分配   —— 几乎零算力代价，折损常常直接是 0，**性价比最高，先修**
# 2) ⑤ 数据   —— 决定指标上限；不查清楚，后面所有实验都读不懂
# 3) ② 度量   —— 与 ③ 一起改最省事（尺度自适应阈值 / NWD）
# 4) ④ 架构   —— 加 P2 代价约 4x，但可预期
# 5) ① 物理   —— 最贵（换相机/提分辨率），且常受产品约束，放最后
#
# 反模式：先换更大 backbone -> 再提分辨率 -> 再加 P2。三个动作都在修 ①④，
#         而真正为零的 ③ 一直没人查。**先算 (s/A_min)^2，能省下的时间以人月计。**
'''
print(RECIPE)
for kw in ['min_anchor_needed', 'd_tol', 'best_iou', 'rel_err', '动手顺序', '反模式']:
    assert kw in RECIPE, kw
print()
print('✅ 胶囊覆盖：五笔账的可执行版本 + 动手顺序 + 最常见的反模式')

### 小结

- **IoU 是尺度不变的尺子，但检测系统的误差是尺度不变不了的。**
  `IoU(s,d) = (s-d)²/(2s²-(s-d)²)`，斜率 `dIoU/dd|₀ = -4/s`。
  同样偏 2px：**8×8 框 0.3913，64×64 框 0.8841**。把位移写成相对量 `d=αs` 时 IoU 与 `s` 无关 ——
  **不公平来自「用相对尺子量绝对误差」，不来自 IoU 公式本身。**
- **临界位移 `d* = s(1-√(2τ/(1+τ)))`，τ=0.5 时 ≈ 0.1835·s。**
  同一条 IoU≥0.5，对 8px 目标要求 1.47px、对 64px 要求 11.74px。
  AP@0.75 对 8px 目标只允许 **0.59px** —— 亚像素，比标注精度还高。
  **AP_S 里有一块是「指标定义造成的」，必须与「模型的账」分开记。**
- **正样本稀缺不是「少」，是「零」。** `IoU_max(s,A) = (s/A)²` 是上界；
  RetinaNet 最小 anchor 32px 下，**8px 目标上界只有 0.0625，22.63px 是死线**，
  蒙特卡洛匹配率精确地是 **0.000**。修法优先级最高且几乎免费。
- **stride 32 上 8px 目标只占 0.25 个格子（面积 0.0625 格）**，输入像素预算差 64 倍；
  加 P2 让 neck+head 空间代价 **×4.0**，且收益只落在最小的桶。
- **有效感受野 σ = √(2n/3)，理论感受野按 O(n)、有效感受野按 O(√n)。**
  σ=30px 时 8px 目标只占该神经元有效输入的 **1.1%**、64px 占 **51.0%**（差 45 倍）。
  所以**「小目标需要更大感受野」是反的** —— 需要的是**匹配**的感受野。
- **±2px 标注抖动对 8px 框是 25% 相对误差**：平均一致性 IoU 只有 **0.62**，
  **AP@0.75 的天花板只有 8.3%**。这一条与前四条**完全正交**，模型侧怎么改都无效。
- **五个原因是乘性的**：`AP_S/AP_L ≈ Πrᵢ`。所以单点改进收益总低于预期，
  且修好一项后瓶颈会立刻换到下一项（练习 4 里「加 P2 后一米都没多看到」就是活例）。
  **诊断顺序：③分配 → ⑤数据 → ②度量 → ④架构 → ①物理。**

下一站：**模块 02 · 架构层面的解法** —— 把原因 ④ 这笔账花在刀刃上。